# Homework 03 — Direct Polynomial Interpolation

## Problem

Consider a set of transit observations stored in the file `data.npy`. The dataset has shape $(196,3)$ and contains the following columns:

1. Time measured in days from mid-transit.
2. Relative flux with observational noise.
3. Flux uncertainty, with a constant value of $1.1\times10^{-4}$.

The objective is to estimate the relative flux at

$$
t=-0.0025\ \text{days},
$$

using direct polynomial interpolation between data points close to the transit minimum.

The interpolation polynomial is obtained by solving the linear system

$$
V\mathbf{a}=\mathbf{y},
$$

where $V$ is the Vandermonde matrix constructed from the selected time values,

$$
V_{ij}=x_i^j,
$$

and $\mathbf{a}$ contains the coefficients of the interpolation polynomial

$$
P(x)=a_0+a_1x+a_2x^2+\cdots+a_{n-1}x^{n-1}.
$$

## Tasks

**a)** Load the data from `data.npy` and identify the time, relative flux, and flux uncertainty columns.

In [8]:
import numpy as np

data = np.load("data.npy")
print(data.shape)
print(data[:5])

(196, 3)
[[-9.00000000e-02  1.00005464e+00  1.10000000e-04]
 [-8.90740741e-02  9.99984791e-01  1.10000000e-04]
 [-8.81481481e-02  1.00007125e+00  1.10000000e-04]
 [-8.72222222e-02  1.00016753e+00  1.10000000e-04]
 [-8.62962963e-02  9.99974243e-01  1.10000000e-04]]


**b)** Select the data points required to perform the interpolation near $t=-0.0025$ days.

In [21]:
#Punto b
t = data[:,0]
flujo = data[:,1]
incertidumbre = data[:,2]
indice_minimo = np.argmin(flujo)

print(f"El indice donde el flujo es minimo es: {indice_minimo}")
print(f"El tiempo donde el flujo es minimo es: {t[indice_minimo]}")
print(f"El flujo minimo es: {flujo[indice_minimo]}")

x = t[94:96]
y = flujo[94:96]
x_eval = -0.0025

print(f"Puntos seleccionados en tiempo: {x}")
print(f"Puntos seleccionados en flujo: {y}")
print(f"Punto donde se va a interpolar: {x_eval}")

El indice donde el flujo es minimo es: 95
El tiempo donde el flujo es minimo es: -0.0020370370370369068
El flujo minimo es: 0.9820790294159998
Puntos seleccionados en tiempo: [-0.00296296 -0.00203704]
Puntos seleccionados en flujo: [0.98220196 0.98207903]
Punto donde se va a interpolar: -0.0025


**c)** Implement a function to construct the Vandermonde matrix without using a built-in Vandermonde function.

In [15]:
#Punto c
def vandermonde(x_datos):
  n = len(x_datos)
  V = np.zeros((n,n))

  for i in range(n):
    for j in range(n):
      V[i][j] = x_datos[i]**j

  return V

V = vandermonde(x)
print(V)

[[ 1.         -0.00296296]
 [ 1.         -0.00203704]]


**d)** Implement Gaussian elimination with partial pivoting to solve the linear system for the polynomial coefficients.

In [16]:
#Punto d
def gauss_solve(A,b):
  n = len(b)
  M = np.column_stack((A.copy(),b.copy()))

  for k in range(n):
    piv = k + np.argmax(np.abs(M[k:,k]))
    M[[k,piv]] = M[[piv,k]]

    for i in range(k+1,n):
      factor = M[i][k]/M[k][k]

      for j in range(k,n+1):
        M[i][j] = M[i][j] - factor*M[k][j]

  x = np.zeros(n)

  for i in range(n-1,-1,-1):
    suma = 0

    for j in range(i+1,n):
      suma = suma + M[i][j]*x[j]

    x[i] = (M[i][n]-suma)/M[i][i]

  return x

a = gauss_solve(V,y)

print(f"Coeficientes del polinomio: {a}")

Coeficientes del polinomio: [ 0.98180859 -0.13275994]


**e)** Implement a direct polynomial interpolation function using the Vandermonde matrix and Gaussian elimination.

In [18]:
#Punto e
def interp_directo(x_datos,y_datos,x_eval):
  V = vandermonde(x_datos)
  a = gauss_solve(V,y_datos)

  resultado = 0

  for j in range(len(a)):
    resultado = resultado + a[j]*(x_eval**j)

  return resultado

**f)** Estimate the relative flux at

$$
t=-0.0025\ \text{days}.
$$

In [19]:
#Punto f
flujo_interpolado = interp_directo(x,y,x_eval)

print(f"El flujo interpolado en t = {x_eval} dias es: {flujo_interpolado}")

El flujo interpolado en t = -0.0025 dias es: 0.9821404923515868


## Conclusion

Using direct polynomial interpolation with the Vandermonde matrix and Gaussian elimination, the estimated relative flux at $t=-0.0025$ days is

$$
F(-0.0025) \approx 0.98214049.
$$

This value lies between the flux values of the two neighboring observations, as expected for an interpolation between these data points.